# Exploratory Data Analysis (EDA): Green Taxi Trip Data (Jan 2021)

Data:  `green_tripdata_2021-01.parquet`. 

Goal: 
- Understand the data structure
- Identify missing or inconsistent values
- Extract preliminary insights

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
data_path = "../data/green_tripdata_2021-01.parquet"
df = pd.read_parquet(data_path)

---

In [ ]:
print("="*50)
print(f"Shape: {df.shape}")
print("="*50)

print("\nDATA TYPES:")
print("-" * 30)
print(df.dtypes)

print("\nHEAD")
print("-" * 30)
display(df.head(5))

---
** Missing Values **

In [ ]:
# percentage of nulls by column
null_counts = df.isnull().sum()
null_ratio = (null_counts / df.shape[0]) * 100

# missing data Dataframe
missing_data = pd.DataFrame({'Missing Values': null_counts, '(%)': null_ratio})
missing_data = missing_data[missing_data['Missing Values'] > 0].sort_values(by='(%)', ascending=False)

print("="*50)
print("MISSING VALUES SUMMARY")
print("="*50)
display(missing_data.round(2))

---
** Descriptive Statistics **

In [ ]:
print("="*50)
print("DESCRIPTIVE STATISTICS (Quant. Variables)")
print("="*50)

# Selecting quantitative features
quant_features = [
    'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
    'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 
    'congestion_surcharge'
]
df_describe = df[quant_features].describe()
display(df_describe.round(2))

# possible anomaly check
print("\n POSSIBLE ANOMALY CHECK:")
print("-" * 30)
possible_anom = (df[quant_features] <= 0).sum()
df_possible_anom = pd.DataFrame({'Count <= 0': possible_anom.values}, index=possible_anom.index)
display(df_possible_anom)

---
** Quick possible anomaly filtering (No deep analysis!) **

In [ ]:
# filters
passenger_count = df['passenger_count'] > 0
trip_distance   = df['trip_distance'] > 0
fare_amount     = df['fare_amount'] > 0
total_amount    = df['total_amount'] > 0

df_filtered = df[
    passenger_count & \
    trip_distance   & \
    fare_amount
]

print("="*50)
print(f"FILTERED DATASET SHAPE: {df_filtered.shape}")



---
** simple feature eng **

In [ ]:
# Ensuring datetime objects
df_filtered['lpep_pickup_datetime'] = pd.to_datetime(df_filtered['lpep_pickup_datetime'])
df_filtered['lpep_dropoff_datetime'] = pd.to_datetime(df_filtered['lpep_dropoff_datetime'])

# Calculating duration in minutes
df_filtered['duration_minutes'] = (df_filtered['lpep_dropoff_datetime'] - df_filtered['lpep_pickup_datetime']).dt.total_seconds() / 60

# Spliting temporal information
df_filtered['pickup_hour'] = df_filtered['lpep_pickup_datetime'].dt.hour
df_filtered['pickup_day_name'] = df_filtered['lpep_pickup_datetime'].dt.day_name()

print("="*50)
print("NEW FEATURES CREATED: 'duration_minutes', 'pickup_hour', 'pickup_name'")
print("="*50)
display(df_filtered[['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'duration_minutes', 'pickup_hour', 'pickup_day_name']].head())

---
** Some Datavis **

In [ ]:
from matplotlib.gridspec import GridSpec

# Define the chronological order for the days of the week
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Create a figure with a custom layout using GridSpec
# 3 rows and 2 columns
fig = plt.figure(figsize=(18, 18))
gs = GridSpec(3, 2, figure=fig, hspace=0.4, wspace=0.2)

# Define the 6 axes
ax1 = fig.add_subplot(gs[0, 0]) # Row 0, Col 0 (Hour - Volume)
ax2 = fig.add_subplot(gs[0, 1]) # Row 0, Col 1 (Hour - Avg Amount)
ax3 = fig.add_subplot(gs[1, 0]) # Row 1, Col 0 (Day - Volume)
ax4 = fig.add_subplot(gs[1, 1]) # Row 1, Col 1 (Day - Avg Amount)
ax5 = fig.add_subplot(gs[2, 0]) # Row 2, Col 0 (Trip Type - Avg Amount)
ax6 = fig.add_subplot(gs[2, 1]) # Row 2, Col 1 (Correlation Matrix)

# ---------------------------------------------------------
# 1. Histogram: Trips by Hour of the Day (Top Left)
# ---------------------------------------------------------
sns.countplot(
    data=df_filtered, 
    x='pickup_hour', 
    color='mediumseagreen', 
    ax=ax1
)
ax1.set_title('Trip Volume by Hour of the Day', fontsize=14, pad=10)
ax1.set_xlabel('Hour of the Day (0-23)')
ax1.set_ylabel('Number of Trips')

# ---------------------------------------------------------
# 2. Average Fare Amount by Hour of the Day (Top Right)
# ---------------------------------------------------------
sns.barplot(
    data=df_filtered, 
    x='pickup_hour', 
    y='fare_amount', 
    color='cornflowerblue', 
    errorbar=None, 
    ax=ax2
)
ax2.set_title('Average Fare Amount by Hour of the Day', fontsize=14, pad=10)
ax2.set_xlabel('Hour of the Day (0-23)')
ax2.set_ylabel('Average Fare Amount ($)')

# ---------------------------------------------------------
# 3. Histogram: Trips by Day of the Week (Middle Left)
# ---------------------------------------------------------
sns.countplot(
    data=df_filtered, 
    x='pickup_day_name', 
    order=days_order, 
    palette='viridis', 
    hue='pickup_day_name',
    legend=False, 
    ax=ax3
)
ax3.set_title('Trip Volume by Day of the Week', fontsize=14, pad=10)
ax3.set_xlabel('Day of the Week')
ax3.set_ylabel('Number of Trips')
ax3.tick_params(axis='x', rotation=15)

# ---------------------------------------------------------
# 4. Average Fare Amount by Day of the Week (Middle Right)
# ---------------------------------------------------------
sns.barplot(
    data=df_filtered, 
    x='pickup_day_name', 
    y='fare_amount', 
    order=days_order, 
    palette='magma', 
    hue='pickup_day_name',
    legend=False,
    errorbar=None, 
    ax=ax4
)
ax4.set_title('Average Fare Amount by Day of the Week', fontsize=14, pad=10)
ax4.set_xlabel('Day of the Week')
ax4.set_ylabel('Average Fare Amount ($)')
ax4.tick_params(axis='x', rotation=15)

# ---------------------------------------------------------
# 5. Average Fare Amount by Trip Type (Bottom Left)
# ---------------------------------------------------------
sns.barplot(
    data=df_filtered, 
    x='trip_type', 
    y='fare_amount', 
    palette='Set2', 
    hue='trip_type',
    legend=False,
    errorbar=None, 
    ax=ax5
)
ax5.set_title('Average Fare Amount by Trip Type', fontsize=14, pad=10)
ax5.set_xlabel('Trip Type (1 = Street-hail, 2 = Dispatch)')
ax5.set_ylabel('Average Fare Amount ($)')

# ---------------------------------------------------------
# 6. Correlation Matrix (Bottom Right)
# ---------------------------------------------------------
corr_cols = ['trip_distance', 'fare_amount', 'tip_amount', 'total_amount', 'duration_minutes', 'congestion_surcharge', 'improvement_surcharge']
corr_matrix = df_filtered[corr_cols].corr()
sns.heatmap(
    corr_matrix, 
    annot=True, 
    cmap='coolwarm', 
    fmt=".2f", 
    vmin=-1, 
    vmax=1, 
    ax=ax6
)
ax6.set_title('Correlation Matrix of Numeric Features', fontsize=14, pad=10)

plt.tight_layout()
plt.show()

print("="*50)

In [ ]:
outlier_hour = df_filtered['pickup_hour'] == 5
outlier_hour_mean = df_filtered[outlier_hour]['fare_amount'].mean()
outlier_hour_std = df_filtered[outlier_hour]['fare_amount'].std()
superior_lim = df_filtered[outlier_hour]['fare_amount'] > (outlier_hour_mean + outlier_hour_std)
df_filtered[outlier_hour][superior_lim]['duration_minutes'].hist(bins=20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ---------------------------------------------------------
# 1. Histogram of Fare Amount (Left)
# ---------------------------------------------------------
sns.histplot(
    data=df_filtered, 
    x='fare_amount', 
    bins=50, 
    kde=True, 
    color='purple',
    ax=axes[0]
)
axes[0].set_title('Distribution of Fare Amount', fontsize=14, pad=10)
axes[0].set_xlabel('Fare Amount ($)', fontsize=12)
axes[0].set_ylabel('Frequency (Number of Trips)', fontsize=12)

# ---------------------------------------------------------
# 2. Box-plot of Fare Amount (Right)
# ---------------------------------------------------------
sns.boxplot(
    data=df_filtered, 
    x='fare_amount', 
    color='mediumorchid',
    ax=axes[1]
)
axes[1].set_title('Spread and Outliers of Fare Amount', fontsize=14, pad=10)
axes[1].set_xlabel('Fare Amount ($)', fontsize=12)

plt.tight_layout()
plt.show()

print("="*50)


In [ ]:
from matplotlib.gridspec import GridSpec

# Define the chronological order for the days of the week
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Create a figure with a custom layout using GridSpec
# Expanded to 4 rows and 2 columns
fig = plt.figure(figsize=(18, 24))
gs = GridSpec(4, 2, figure=fig, hspace=0.4, wspace=0.2)

# Define the 8 axes
ax1 = fig.add_subplot(gs[0, 0]) # Row 0, Col 0 (Hour - Volume)
ax2 = fig.add_subplot(gs[0, 1]) # Row 0, Col 1 (Hour - Avg Tip)
ax3 = fig.add_subplot(gs[1, 0]) # Row 1, Col 0 (Day - Volume)
ax4 = fig.add_subplot(gs[1, 1]) # Row 1, Col 1 (Day - Avg Tip)
ax5 = fig.add_subplot(gs[2, 0]) # Row 2, Col 0 (Trip Type - Avg Tip)
ax6 = fig.add_subplot(gs[2, 1]) # Row 2, Col 1 (Payment Type - Volume)
ax7 = fig.add_subplot(gs[3, 0]) # Row 3, Col 0 (Payment Type - Avg Tip)
ax8 = fig.add_subplot(gs[3, 1]) # Row 3, Col 1 (Correlation Matrix)

# ---------------------------------------------------------
# 1. Histogram: Trips by Hour of the Day (Top Left)
# ---------------------------------------------------------
sns.countplot(
    data=df_filtered, 
    x='pickup_hour', 
    color='mediumseagreen', 
    ax=ax1
)
ax1.set_title('Trip Volume by Hour of the Day', fontsize=14, pad=10)
ax1.set_xlabel('Hour of the Day (0-23)')
ax1.set_ylabel('Number of Trips')

# ---------------------------------------------------------
# 2. Average Tip Amount by Hour of the Day (Top Right)
# ---------------------------------------------------------
sns.barplot(
    data=df_filtered, 
    x='pickup_hour', 
    y='tip_amount', 
    color='cornflowerblue', 
    errorbar=None, 
    ax=ax2
)
ax2.set_title('Average Tip Amount by Hour of the Day', fontsize=14, pad=10)
ax2.set_xlabel('Hour of the Day (0-23)')
ax2.set_ylabel('Average Tip Amount ($)')

# ---------------------------------------------------------
# 3. Histogram: Trips by Day of the Week (Row 2 Left)
# ---------------------------------------------------------
sns.countplot(
    data=df_filtered, 
    x='pickup_day_name', 
    order=days_order, 
    palette='viridis', 
    hue='pickup_day_name',
    legend=False, 
    ax=ax3
)
ax3.set_title('Trip Volume by Day of the Week', fontsize=14, pad=10)
ax3.set_xlabel('Day of the Week')
ax3.set_ylabel('Number of Trips')
ax3.tick_params(axis='x', rotation=15)

# ---------------------------------------------------------
# 4. Average Tip Amount by Day of the Week (Row 2 Right)
# ---------------------------------------------------------
sns.barplot(
    data=df_filtered, 
    x='pickup_day_name', 
    y='tip_amount', 
    order=days_order, 
    palette='magma', 
    hue='pickup_day_name',
    legend=False,
    errorbar=None, 
    ax=ax4
)
ax4.set_title('Average Tip Amount by Day of the Week', fontsize=14, pad=10)
ax4.set_xlabel('Day of the Week')
ax4.set_ylabel('Average Tip Amount ($)')
ax4.tick_params(axis='x', rotation=15)

# ---------------------------------------------------------
# 5. Average Tip Amount by Trip Type (Row 3 Left)
# ---------------------------------------------------------
sns.barplot(
    data=df_filtered, 
    x='trip_type', 
    y='tip_amount', 
    palette='Set2', 
    hue='trip_type',
    legend=False,
    errorbar=None, 
    ax=ax5
)
ax5.set_title('Average Tip Amount by Trip Type', fontsize=14, pad=10)
ax5.set_xlabel('Trip Type (1 = Street-hail, 2 = Dispatch)')
ax5.set_ylabel('Average Tip Amount ($)')

# ---------------------------------------------------------
# 6. Histogram: Trips by Payment Type (Row 3 Right)
# ---------------------------------------------------------
sns.countplot(
    data=df_filtered, 
    x='payment_type', 
    palette='Set1', 
    hue='payment_type',
    legend=False, 
    ax=ax6
)
ax6.set_title('Trip Volume by Payment Type', fontsize=14, pad=10)
ax6.set_xlabel('Payment Type (1=Credit, 2=Cash, 3=No Charge, 4=Dispute)')
ax6.set_ylabel('Number of Trips')

# ---------------------------------------------------------
# 7. Average Tip Amount by Payment Type (Row 4 Left)
# ---------------------------------------------------------
sns.barplot(
    data=df_filtered, 
    x='payment_type', 
    y='tip_amount', 
    palette='Set1', 
    hue='payment_type',
    legend=False,
    errorbar=None, 
    ax=ax7
)
ax7.set_title('Average Tip Amount by Payment Type', fontsize=14, pad=10)
ax7.set_xlabel('Payment Type (1=Credit, 2=Cash, 3=No Charge, 4=Dispute)')
ax7.set_ylabel('Average Tip Amount ($)')

# ---------------------------------------------------------
# 8. Correlation Matrix (Row 4 Right)
# ---------------------------------------------------------
corr_cols = ['trip_distance', 'fare_amount', 'tip_amount', 'total_amount', 'duration_minutes', 'congestion_surcharge', 'improvement_surcharge']
corr_matrix = df_filtered[corr_cols].corr()
sns.heatmap(
    corr_matrix, 
    annot=True, 
    cmap='coolwarm', 
    fmt=".2f", 
    vmin=-1, 
    vmax=1, 
    ax=ax8
)
ax8.set_title('Correlation Matrix of Numeric Features', fontsize=14, pad=10)

plt.tight_layout()
plt.show()

print("="*50)

# =========================================================
# SEPARATED FIGURE: DISTRIBUTION AND SPREAD
# =========================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ---------------------------------------------------------
# 1. Histogram of Tip Amount (Left)
# ---------------------------------------------------------
sns.histplot(
    data=df_filtered, 
    x='tip_amount', 
    bins=50, 
    kde=True, 
    color='purple',
    ax=axes[0]
)
axes[0].set_title('Distribution of Tip Amount', fontsize=14, pad=10)
axes[0].set_xlabel('Tip Amount ($)', fontsize=12)
axes[0].set_ylabel('Frequency (Number of Trips)', fontsize=12)

# ---------------------------------------------------------
# 2. Box-plot of Tip Amount (Right)
# ---------------------------------------------------------
sns.boxplot(
    data=df_filtered, 
    x='tip_amount', 
    color='mediumorchid',
    ax=axes[1]
)
axes[1].set_title('Spread and Outliers of Tip Amount', fontsize=14, pad=10)
axes[1].set_xlabel('Tip Amount ($)', fontsize=12)

plt.tight_layout()
plt.show()

print("="*50)

---
** potential features for *_Tip Amount regression_* and *_Payment Type cls_* (quick analysis)**


In [ ]:
potential_features = [
    'pickup_hour',    
    'pickup_day_name',
    'duration_minutes',
    'trip_distance',
    'trip_type',
    'congestion_surcharge',
    'fare_amount',
    'PULocationID',
    'DOLocationID',
    'tip_amount',
    'payment_type',
]

df_selected = df_filtered[potential_features]

In [ ]:
# percentage of nulls by column
null_counts = df_selected.isnull().sum()
null_ratio = (null_counts / df_selected.shape[0]) * 100

# missing data Dataframe
missing_data = pd.DataFrame({'Missing Values': null_counts, '(%)': null_ratio})
missing_data = missing_data[missing_data['Missing Values'] > 0].sort_values(by='(%)', ascending=False)

print("="*50)
print("MISSING VALUES SUMMARY")
print("="*50)
display(missing_data.round(2))

In [ ]:
print(df_selected.shape)
display(df_selected.head())

In [ ]:
potential_features

In [ ]:
print("="*50)
print("DESCRIPTIVE STATISTICS (Quant. Variables)")
print("="*50)

# Selecting quantitative features
quant_features = [
    'duration_minutes', 'congestion_surcharge', 'trip_distance', 'fare_amount', 'tip_amount'
]
df_describe = df_selected[quant_features].describe()
display(df_describe.round(2))

# possible anomaly check
print("\n POSSIBLE ANOMALY CHECK:")
print("-" * 30)
possible_anom = (df_selected[quant_features] < 0).sum()
df_possible_anom = pd.DataFrame({'Count < 0': possible_anom.values}, index=possible_anom.index)
display(df_possible_anom)

---
** export training data **

In [ ]:
training_data_path = '../data/training_taxi_data.parquet'
df_filtered.to_parquet(training_data_path, index=False)